In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import numpy as np

FUNCTION-FUNCTION PENTING

In [ ]:
# 1. Fungsi Cleaning (pembersihan spasi) Genre
def clean_genres(genre_string):
    if pd.isna(genre_string) or not isinstance(genre_string, str):
        return []
    # Ganti '&' dan ',' dengan spasi, ubah ke lowercase
    cleaned = genre_string.lower().replace('&', ' ').replace(',', ' ')
    words = cleaned.split()
    # Buang kata-kata sampah yang berulang
    stop_words = {'tv', 'shows', 'movies', 'show', 'movie'}
    filtered_words = [w for w in words if w not in stop_words]
    return filtered_words

In [ ]:
# 2. Fungsi Cek Semua Genre Unik di Dataset
def get_all_unique_genres(dataframe):
    unique_genres = set()
    for listed_in in dataframe['listed_in'].dropna():
        unique_genres.update(clean_genres(listed_in))
    return sorted(list(unique_genres))

In [11]:

# 3. Fungsi Batasan Rating Umur (Sama seperti bawaanmu)
def get_max_allowed_rating(age):
    if age < 7: return ['G', 'TV-Y', 'TV-G']
    elif age < 13: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG']
    elif age < 14: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG', 'PG-13']
    elif age < 17: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG', 'PG-13', 'TV-14']
    else: return ['G', 'TV-Y', 'TV-G', 'PG', 'TV-Y7', 'TV-Y7-FV', 'TV-PG', 'PG-13', 'TV-14', 'R', 'NC-17', 'TV-MA', 'NR', 'UR']

In [ ]:
# 4. Fungsi Utama Rekomendasi (Mengatasi Cold-Start & Fallback)
def get_recommendations(user_data, df_catalog, embeddings_matrix, top_n=5):
    age = user_data.get('age', 18)
    watch_history = user_data.get('watch_history', [])
    preferred_genres = user_data.get('preferred_genres', [])
    
    # Filter rating umur di awal agar aman
    allowed_ratings = get_max_allowed_rating(age)
    df_scores = df_catalog.copy()
    df_scores['rating'] = df_scores['rating'].fillna('NR')
    df_scores = df_scores[df_scores['rating'].isin(allowed_ratings)].reset_index(drop=True)
    
    # Ambil ulang matriks embedding yang sudah difilter berdasarkan indeks rating umur
    # (indeks catalog yang valid sesuai rating umur)
    valid_indices = df_catalog[df_catalog['rating'].fillna('NR').isin(allowed_ratings)].index
    filtered_embeddings = embeddings_matrix[valid_indices]
    
    # Bersihkan preferensi genre user (abaikan kata yang tidak ada di list token unik)
    catalog_genres = get_all_unique_genres(df_catalog)
    clean_user_pref = [w for genre in preferred_genres for w in clean_genres(genre) if w in catalog_genres]
    
    # Cari indeks film history di dataframe yang sudah difilter umur
    history_titles = [m['title'].lower() for m in watch_history]
    history_indices_in_filtered = df_scores[df_scores['title'].str.lower().isin(history_titles)].index.tolist()
    
    # --- LOGIKA REKOMENDASI & HANDLING COLD-START ---
    if not watch_history or not history_indices_in_filtered:
        # TANTANGAN: COLD START (History Kosong / Film Tidak Ditemukan)
        # Fallback: Buat profile vector berdasarkan 'preferred_genres' murni menggunakan model text
        if clean_user_pref:
            pref_text = " ".join(clean_user_pref)
            # Encode teks preferensi menggunakan model global yang sedang aktif
            user_profile_vector = model.encode([pref_text]).reshape(1, -1)
            scores = cosine_similarity(user_profile_vector, filtered_embeddings)[0]
        else:
            # Jika preferred_genres juga kosong/tidak valid, fallback ke item terbaru di katalog yang aman
            return df_scores.sort_values('release_year', ascending=False).head(top_n)
    else:
        # KONDISI NORMAL: Ambil rata-rata embedding dari history nonton
        user_vecs = filtered_embeddings[history_indices_in_filtered]
        user_profile_vector = np.mean(user_vecs, axis=0).reshape(1, -1)
        scores = cosine_similarity(user_profile_vector, filtered_embeddings)[0]
    
    df_scores['similarity_score'] = scores
    
    # Hapus film yang sudah ditonton agar tidak direkomendasikan lagi
    df_final = df_scores[~df_scores['title'].str.lower().isin(history_titles)]
    return df_final.sort_values('similarity_score', ascending=False).head(top_n)

In [13]:
def evaluate_recommendations(recommendations, user_data, df_catalog):
    if recommendations.empty:
        return { "hit_rate": 0, "f1_score": 0, "precision_at_k": 0, "dcg": 0.0, "ndcg": 0.0, "relevance_scores": [], "ideal_relevance_scores": [] }

    # ── Bangun kumpulan kata target dari preferred genres + watch history ──
    user_target_words = set()
    for g in user_data.get('preferred_genres', []):
        user_target_words.update(clean_genres(g))

    for item in user_data.get('watch_history', []):
        match = df_catalog[df_catalog['title'].str.lower() == item['title'].lower()]
        if not match.empty:
            row = match.iloc[0]
            user_target_words.update(clean_genres(row['listed_in']))
            if row['director']:
                user_target_words.update(row['director'].lower().replace(',', ' ').split())
            if row['cast']:
                user_target_words.update(row['cast'].lower().replace(',', ' ').split())

    hits = 0
    f1_scores = []
    relevance_scores = []   # Graded relevance per posisi (0-3 scale)

    for _, row in recommendations.iterrows():
        rec_words = set(clean_genres(row['listed_in']))
        if row['director']:
            rec_words.update(row['director'].lower().replace(',', ' ').split())
        if row['cast']:
            rec_words.update(row['cast'].lower().replace(',', ' ').split())

        intersection = user_target_words.intersection(rec_words)

        # ── Hit ──
        if len(intersection) > 0:
            hits += 1

        # ── F1 ──
        if len(rec_words) == 0 or len(user_target_words) == 0:
            f1 = 0.0
        else:
            precision = len(intersection) / len(rec_words)
            recall    = len(intersection) / len(user_target_words)
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)

        # ── Graded Relevance (0-3) berdasarkan overlap ratio ──
        if len(user_target_words) == 0:
            rel = 0
        else:
            overlap_ratio = len(intersection) / len(user_target_words)
            if overlap_ratio >= 0.15:
                rel = 3
            elif overlap_ratio >= 0.08:
                rel = 2
            elif overlap_ratio > 0:
                rel = 1
            else:
                rel = 0
        relevance_scores.append(rel)

    k = len(relevance_scores)

    # ── Precision@K ──
    precision_at_k = hits / k if k > 0 else 0.0

    # ── DCG  (formula 9 dari slide: rel_i / log2(i+1)) ──
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevance_scores))

    # ── IDCG – ideal ordering (sort descending) ──
    ideal_rels = sorted(relevance_scores, reverse=True)
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels))

    # ── NDCG ──
    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    hit_rate = 1 if hits > 0 else 0

    return {
        "hit_rate":              hit_rate,
        "f1_score":              round(float(np.mean(f1_scores)), 4),
        "precision_at_k":        round(float(precision_at_k), 4),
        "dcg":                   round(float(dcg), 4),
        "ndcg":                  round(float(ndcg), 4),
        "relevance_scores":      relevance_scores,
        "ideal_relevance_scores": ideal_rels,
    }

LOAD DATA

In [14]:
df = pd.read_csv('netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


PRE-PROCESSING DATA

In [16]:
# Ambil kolom yang dibutuhkan dan tangani NaN
required_features = ['title', 'rating', 'listed_in', 'description', 'director', 'cast']
for feature in required_features:
    df[feature] = df[feature].fillna('')

# Bikin kolom teks gabungan dengan menerapkan pembersihan listed_in langsung di stringnya
def build_combined_text(row):
    # Ganti nama variabel lokal menjadi 'genres_text' atau 'genres_cleaned'
    genres_text = " ".join(clean_genres(row['listed_in']))
    
    # Gunakan variabel baru tersebut di f-string
    return f"{row['title']}{row['director']}{row['cast']}{genres_text} {row['description']}".lower()

df['combined_features'] = df.apply(build_combined_text, axis=1)
print("Data Preprocessing Selesai. Kolom 'combined_features' siap di-embed.")

Data Preprocessing Selesai. Kolom 'combined_features' siap di-embed.


In [17]:
# Ambil seluruh genre pada 'listed_in' untuk referensi (bisa digunakan untuk validasi preferensi genre user)
all_genres = get_all_unique_genres(df)
print(f"Total genre unik yang ditemukan: {len(all_genres)}")
print("Daftar genre unik:")
for genre in all_genres:
    print("-", genre, "\b")

Total genre unik yang ditemukan: 39
Daftar genre unik:
- action
- adventure
- anime
- british
- children
- classic
- comedies
- comedy
- crime
- cult
- documentaries
- docuseries
- dramas
- faith
- family
- fantasy
- features
- horror
- independent
- international
- kids'
- korean
- lgbtq
- music
- musicals
- mysteries
- nature
- reality
- romantic
- sci-fi
- science
- series
- spanish-language
- spirituality
- sports
- stand-up
- talk
- teen
- thrillers


INISIALISASI METODE EMBEDDING (SBERT)

In [18]:
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

print("Generating embeddings... Mohon tunggu.")
embeddings = model.encode(df['combined_features'].tolist(), show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)

Generating embeddings... Mohon tunggu.


Batches:   0%|          | 0/276 [00:00<?, ?it/s]

Embeddings shape: (8807, 384)


Pengujian dengan mock metadata

In [ ]:
users_metadata = {
    "U001": {
        "name": "Arya Kusuma",
        "age": 24,
        "preferred_genres": ["Thrillers", "Sci-Fi & Fantasy"],
        "watch_history": [
            {"title": "Bird Box"},
            {"title": "The Platform"},
            {"title": "Black Mirror: Bandersnatch"},
            {"title": "Inception"},
            {"title": "Annihilation"},
        ]
    },
    "U002": {
        "name": "Siti Rahayu",
        "age": 31,
        "preferred_genres": ["Romantic Movies", "Comedies"],
        "watch_history": [
            {"title": "To All the Boys I've Loved Before"},
            {"title": "Always Be My Maybe"},
        ]
    },
    "U003": {
        "name": "Budi Santoso",
        "age": 28,
        "preferred_genres": ["Dramas", "International Movies"],
        "watch_history": [
            {"title": "Gie"},
            {"title": "Srimulat: Hil yang Mustahal"},
            {"title": "Ali & Ratu Ratu Queens"},
            {"title": "Agak Laen"},  
        ]
    },
    "U004": {
        "name": "Dewi Lestari",
        "age": 27,
        "preferred_genres": ["Horror Movies", "Thrillers"],
        "watch_history": [
            {"title": "Siksa Kubur"},
            {"title": "KKN di Desa Penari"},
            {"title": "Pengabdi Setan 2: Communion"},
        ]
    },
    "U005": {
        "name": "Reza Firmansyah",
        "age": 19,
        "preferred_genres": ["Action & Adventure", "Otomotif Karburator Honda"],
        "watch_history": [
            {"title": "John Wick"},
            {"title": "Extraction"},
            {"title": "The Old Guard"},
            {"title": "6 Underground"},
        ]
    },
    "U006": {
        "name": "Nia Permata",
        "age": 35,
        "preferred_genres": ["Documentaries", "Kriptografi Quantum Cyber"],
        "watch_history": [
            {"title": "Our Planet"},
            {"title": "Making a Murderer"},
        ]
    },
    "U007": {
        "name": "Farhan Nugroho",
        "age": 29,
        "preferred_genres": ["Comedies", "Budidaya Ikan Lele Kolam Terpal"], 
        "watch_history": [
            {"title": "Ancika: Dia yang Bersamaku 1995"}, 
            {"title": "Petualangan Sherina 2"}, 
            {"title": "Pasutri Gaje"},
            {"title": "Kaka Boss"}, 
        ]
    },
    # 8. Genre Sebagian Ada, History Watch Sedikit TAPI TIDAK ADA di Dataset
    "U008": {
        "name": "Maya Indah",
        "age": 22,
        "preferred_genres": ["Children & Family Movies", "Resep Seblak Ceker Pedas"], 
        "watch_history": [
            {"title": "Badarawuhi di Desa Penari"}, # Tidak Ada
            {"title": "Vina: Sebelum 7 Hari"},       # Tidak Ada
        ]
    },
    # 9. TARGET METRIK RENDAH: Genre Semuanya TIdak Ada (Sangat Absurd), History Watch Banyak & Terdaftar
    "U009": {
        "name": "Hendra Wijaya",
        "age": 45,
        "preferred_genres": ["Mesin Jahit Konveksi", "Suku Cadang Mesin Diesel Diesel"],
        "watch_history": [
            {"title": "Inception"},
            {"title": "Interstellar"},
            {"title": "The Matrix"},
            {"title": "Blade Runner 2049"},
        ]
    },
    # 10. TARGET METRIK RENDAH: Genre Semuanya Tidak Ada (Absurd), History Watch Sedikit & Terdaftar
    "U010": {
        "name": "Ratna Sari",
        "age": 33,
        "preferred_genres": ["Pertanian Organik Hidroponik", "Tekstil Industri Kain Katun"],
        "watch_history": [
            {"title": "The Notebook"},
            {"title": "Pride & Prejudice"},
        ]
    },
    # 11. PENGHANCUR METRIK (TARGET HIT RATE = 0 & SIMILARITY < 0.4): 
    # Genre Semuanya Tidak Ada (Absurd), History Banyak TAPI SEMUANYA TIDAK ADA di Dataset.
    "U011": {
        "name": "Dimas Prasetyo",
        "age": 25,
        "preferred_genres": ["Alat Pertukangan Semen Semprot", "Sistem Pipa Pembuangan Lumpur Sidoarjo"],
        "watch_history": [
            {"title": "Sekawan Limo"},            # Tidak Ada
            {"title": "Ipar adalah Maut"},         # Tidak Ada
            {"title": "Jurnal Risa by Risa Saraswati"}, # Tidak Ada
            {"title": "Do You See What I See"},    # Tidak Ada
        ]
    },
    # 12. PENGHANCUR METRIK: Genre Semuanya Tidak Ada (Absurd), History Sedikit & TIDAK ADA di Dataset
    "U012": {
        "name": "Laila Azzahra",
        "age": 30,
        "preferred_genres": ["Manajemen Akuntansi Neraca Saldo", "Kalkulus Integral Turunan Parsial"],
        "watch_history": [
            {"title": "Kang Mak from Pee Mak"}, # Tidak Ada
            {"title": "Bolehkah Sekali Ini Saja Menangis"}, # Tidak Ada
        ]
    }
}

=== HASIL REKOMENDASI UNTUK: Arya Kusuma ===


,title,listed_in,similarity_score
5113,Bright,"Action & Adventure, Sci-Fi & Fantasy",0.721860
3072,Riot,Action & Adventure,0.697993
1584,Ava,"Action & Adventure, Dramas",0.695497
8671,Vincent N Roxxy,"Dramas, Thrillers",0.694902
4028,Into the Badlands,TV Action & Adventure,0.693001



=== HASIL METRIK EVALUASI: Arya Kusuma ===


hit_rate                                1
f1_score                           0.0301
precision_at_k                        1.0
dcg                                2.9485
ndcg                                  1.0
relevance_scores          [1, 1, 1, 1, 1]
ideal_relevance_scores    [1, 1, 1, 1, 1]
Name: Nilai, dtype: object



=== HASIL REKOMENDASI UNTUK: User Baru Cold Start ===


,title,listed_in,similarity_score
2144,GAME ON: A Comedy Crossover Event,"Kids' TV, TV Comedies",0.557825
618,America: The Motion Picture,"Action & Adventure, Comedies",0.554756
3898,Lunatics,"International TV Shows, TV Comedies",0.551750
4570,Hot Date,"Romantic TV Shows, TV Comedies",0.546514
1038,Dancing Angels,"International TV Shows, Romantic TV Shows, TV ...",0.527347



=== HASIL METRIK EVALUASI: User Baru Cold Start ===


hit_rate                                1
f1_score                           0.2445
precision_at_k                        1.0
dcg                                8.8454
ndcg                                  1.0
relevance_scores          [3, 3, 3, 3, 3]
ideal_relevance_scores    [3, 3, 3, 3, 3]
Name: Nilai, dtype: object

In [ ]:

# JALANKAN PENGUJIAN UNTUK USER ARYA (U001)
target_user = users_metadata["U001"]

print(f"=== HASIL REKOMENDASI UNTUK: {target_user['name']} ===")
hasil_recs = get_recommendations(target_user, df, embeddings, top_n=5)
display(hasil_recs[['title', 'listed_in', 'similarity_score']])

print(f"\n=== HASIL METRIK EVALUASI: {target_user['name']} ===")
metrik = evaluate_recommendations(hasil_recs, target_user, df)
df_metrik = pd.Series(metrik, name="Nilai")
display(df_metrik)

print("\n")

# JALANKAN PENGUJIAN UNTUK USER COLD START (U002)
target_user = users_metadata["U002"]

print(f"=== HASIL REKOMENDASI UNTUK: {target_user['name']} ===")
hasil_recs = get_recommendations(target_user, df, embeddings, top_n=5)
display(hasil_recs[['title', 'listed_in', 'similarity_score']])

print(f"\n=== HASIL METRIK EVALUASI: {target_user['name']} ===")
metrik = evaluate_recommendations(hasil_recs, target_user, df)
df_metrik = pd.Series(metrik, name="Nilai")
display(df_metrik)

In [ ]:


# ── Kumpulkan metrik evaluasi semua user ──
all_metrics = {}
for uid, user_data in users_metadata.items():
    recs = get_recommendations(user_data, df, embeddings, top_n=5)
    metrics = evaluate_recommendations(recs, user_data, df)
    all_metrics[user_data['name']] = metrics

names      = list(all_metrics.keys())
hit_rates  = [all_metrics[n]['hit_rate']       for n in names]
f1_scores  = [all_metrics[n]['f1_score']        for n in names]
prec_k     = [all_metrics[n]['precision_at_k']  for n in names]
dcg_vals   = [all_metrics[n]['dcg']             for n in names]
ndcg_vals  = [all_metrics[n]['ndcg']            for n in names]

x      = np.arange(len(names))
width  = 0.18

# ═══════════════════════════════════════════════════════════════════════
# FIGURE 1 — Grouped Bar Chart: semua metrik per user
# ═══════════════════════════════════════════════════════════════════════
fig1, ax1 = plt.subplots(figsize=(18, 6))
fig1.patch.set_facecolor('#0f0f1a')
ax1.set_facecolor('#1a1a2e')

colors = ['#7c3aed', '#2563eb', '#059669', '#d97706', '#dc2626']
labels = ['Hit Rate', 'F1-Score', 'Precision@K', 'DCG (norm)', 'NDCG']

# Normalisasi DCG agar satu skala dengan metrik lainnya (0-1)
max_dcg = max(dcg_vals) if max(dcg_vals) > 0 else 1
dcg_norm = [v / max_dcg for v in dcg_vals]

datasets = [hit_rates, f1_scores, prec_k, dcg_norm, ndcg_vals]

for i, (data, label, color) in enumerate(zip(datasets, labels, colors)):
    bars = ax1.bar(x + i * width - 2 * width, data, width,
                   label=label, color=color, alpha=0.85,
                   edgecolor='white', linewidth=0.4)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                     f'{h:.2f}', ha='center', va='bottom',
                     fontsize=6, color='white', fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=35, ha='right',
                    fontsize=9, color='#e2e8f0')
ax1.set_yticks(np.arange(0, 1.15, 0.2))
ax1.set_yticklabels([f'{v:.1f}' for v in np.arange(0, 1.15, 0.2)],
                    color='#e2e8f0', fontsize=9)
ax1.set_ylim(0, 1.18)
ax1.set_xlabel('User', fontsize=11, color='#e2e8f0', labelpad=8)
ax1.set_ylabel('Nilai Metrik (0–1)', fontsize=11, color='#e2e8f0', labelpad=8)
ax1.set_title('Perbandingan Metrik Evaluasi Rekomendasi per User',
              fontsize=14, color='white', fontweight='bold', pad=14)
ax1.legend(loc='upper right', fontsize=9,
           facecolor='#1a1a2e', edgecolor='#7c3aed', labelcolor='white')
ax1.tick_params(axis='both', colors='#94a3b8')
for spine in ax1.spines.values():
    spine.set_edgecolor('#334155')
ax1.yaxis.grid(True, linestyle='--', alpha=0.3, color='#475569')
ax1.set_axisbelow(True)
plt.tight_layout()
plt.savefig('eval_bar_chart.png', dpi=150, bbox_inches='tight',
            facecolor=fig1.get_facecolor())
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# FIGURE 2 — Pie Charts: Hit Rate & NDCG distribusi
# ═══════════════════════════════════════════════════════════════════════
fig2, axes = plt.subplots(1, 2, figsize=(14, 6))
fig2.patch.set_facecolor('#0f0f1a')

# ── Pie 1: Hit Rate (berhasil vs gagal) ──
hit_count  = sum(hit_rates)
miss_count = len(hit_rates) - hit_count
ax_p1 = axes[0]
ax_p1.set_facecolor('#1a1a2e')
wedges1, texts1, autotexts1 = ax_p1.pie(
    [hit_count, miss_count],
    labels=[f'Hit ({hit_count})', f'Miss ({miss_count})'],
    colors=['#7c3aed', '#dc2626'],
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='#0f0f1a', linewidth=2),
    textprops=dict(color='white', fontsize=11)
)
for at in autotexts1:
    at.set_fontsize(12)
    at.set_fontweight('bold')
ax_p1.set_title('Distribusi Hit Rate\n(Seluruh User)',
                fontsize=12, color='white', fontweight='bold', pad=12)


In [ ]:

# ── Pie 2: NDCG — distribusi kualitas ranking ──
ndcg_perfect = sum(1 for v in ndcg_vals if v == 1.0)
ndcg_high    = sum(1 for v in ndcg_vals if 0.5 <= v < 1.0)
ndcg_low     = sum(1 for v in ndcg_vals if v < 0.5)
ndcg_counts  = [ndcg_perfect, ndcg_high, ndcg_low]
ndcg_labels  = [f'NDCG=1.0 ({ndcg_perfect})', f'0.5≤NDCG<1 ({ndcg_high})', f'NDCG<0.5 ({ndcg_low})']
ndcg_colors  = ['#059669', '#d97706', '#dc2626']
non_zero     = [(c, l, col) for c, l, col in zip(ndcg_counts, ndcg_labels, ndcg_colors) if c > 0]
ax_p2 = axes[1]
ax_p2.set_facecolor('#1a1a2e')
if non_zero:
    wedges2, texts2, autotexts2 = ax_p2.pie(
        [c for c, _, _ in non_zero],
        labels=[l for _, l, _ in non_zero],
        colors=[col for _, _, col in non_zero],
        autopct='%1.1f%%', startangle=140,
        wedgeprops=dict(edgecolor='#0f0f1a', linewidth=2),
        textprops=dict(color='white', fontsize=10)
    )
    for at in autotexts2:
        at.set_fontsize(12)
        at.set_fontweight('bold')
ax_p2.set_title('Distribusi Kualitas Ranking NDCG\n(Seluruh User)',
                fontsize=12, color='white', fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('eval_pie_charts.png', dpi=150, bbox_inches='tight',
            facecolor=fig2.get_facecolor())
plt.show()


In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# FIGURE 3 — Line Chart: tren F1-Score & NDCG antar user
# ═══════════════════════════════════════════════════════════════════════
fig3, ax3 = plt.subplots(figsize=(14, 5))
fig3.patch.set_facecolor('#0f0f1a')
ax3.set_facecolor('#1a1a2e')

short_names = [n.split()[0] for n in names]
ax3.plot(short_names, f1_scores, marker='o', linewidth=2,
         color='#2563eb', label='F1-Score', markersize=7)
ax3.plot(short_names, ndcg_vals, marker='s', linewidth=2,
         color='#7c3aed', label='NDCG', markersize=7)
ax3.fill_between(short_names, f1_scores, alpha=0.15, color='#2563eb')
ax3.fill_between(short_names, ndcg_vals, alpha=0.15, color='#7c3aed')

for i, (f1, nd) in enumerate(zip(f1_scores, ndcg_vals)):
    ax3.annotate(f'{f1:.2f}', (short_names[i], f1),
                 textcoords='offset points', xytext=(0, 8),
                 fontsize=7.5, color='#93c5fd', ha='center')
    ax3.annotate(f'{nd:.2f}', (short_names[i], nd),
                 textcoords='offset points', xytext=(0, -14),
                 fontsize=7.5, color='#c4b5fd', ha='center')

ax3.set_ylim(-0.05, 1.25)
ax3.set_xlabel('User (Nama Depan)', fontsize=11, color='#e2e8f0', labelpad=8)
ax3.set_ylabel('Nilai (0–1)', fontsize=11, color='#e2e8f0', labelpad=8)
ax3.set_title('Tren F1-Score & NDCG Seluruh User',
              fontsize=13, color='white', fontweight='bold', pad=12)
ax3.tick_params(axis='x', colors='#e2e8f0', labelsize=9, rotation=20)
ax3.tick_params(axis='y', colors='#e2e8f0', labelsize=9)
ax3.legend(fontsize=10, facecolor='#1a1a2e', edgecolor='#7c3aed', labelcolor='white')
for spine in ax3.spines.values():
    spine.set_edgecolor('#334155')
ax3.yaxis.grid(True, linestyle='--', alpha=0.3, color='#475569')
ax3.set_axisbelow(True)
plt.tight_layout()
plt.savefig('eval_line_chart.png', dpi=150, bbox_inches='tight',
            facecolor=fig3.get_facecolor())
plt.show()

# ── Tampilkan ringkasan DataFrame ──
df_summary = pd.DataFrame({
    'User'          : names,
    'Hit Rate'      : hit_rates,
    'F1-Score'      : f1_scores,
    'Precision@K'   : prec_k,
    'DCG'           : dcg_vals,
    'NDCG'          : ndcg_vals,
})
print('\n=== RINGKASAN METRIK SEMUA USER ===')
display(df_summary.set_index('User'))


## 📊 Kesimpulan Evaluasi Sistem Rekomendasi (Content-Based Filtering)

Berdasarkan hasil evaluasi terhadap **12 user** dengan berbagai karakteristik (watch history, genre preferensi, dan usia), diperoleh beberapa poin kesimpulan utama:

---

### ✅ Kekuatan Sistem

| Aspek | Penjelasan |
|---|---|
| **Hit Rate Tinggi** | Mayoritas user (terutama U001–U007) mendapatkan hit rate = 1, artinya sistem berhasil merekomendasikan setidaknya satu konten yang relevan. |
| **Precision@K Sempurna** | Pada user dengan watch history yang terdaftar di dataset, Precision@K mencapai **1.0**, menunjukkan semua rekomendasi Top-5 bersifat relevan. |
| **NDCG = 1.0** | Beberapa user mendapat NDCG sempurna, artinya urutan hasil rekomendasi sudah **optimal** sesuai tingkat relevansi ideal. |
| **Cold-Start Ditangani** | User tanpa watch history (cold-start) tetap mendapat rekomendasi bermakna via fallback ke preferred_genres. |

---

### ⚠️ Keterbatasan Sistem

| Aspek | Penjelasan |
|---|---|
| **F1-Score Rendah pada Beberapa User** | User seperti U001 (Arya Kusuma) mendapat F1-Score sangat rendah (~0.03). Ini disebabkan oleh kesenjangan besar antara jumlah kata relevan user vs. jumlah kata unik per konten (precision tinggi, recall rendah). |
| **Genre Tidak Valid Diabaikan** | Preferensi genre yang tidak sesuai kosakata katalog (mis. 'Otomotif Karburator Honda') otomatis diabaikan — sistem tetap berjalan, namun profil user menjadi kurang akurat. |
| **History Tidak Ditemukan = Cold-Start** | User dengan semua judul film tidak ada di dataset (mis. U008, U011, U012) jatuh ke mode cold-start dan kualitas rekomendasinya bergantung sepenuhnya pada genre preferensi. |
| **Hit Rate = 0 pada Kasus Ekstrem** | User dengan genre absurd **dan** history tidak ditemukan (mis. U011, U012) berpotensi mendapat hit rate rendah karena tidak ada informasi valid yang bisa digunakan sebagai sinyal relevansi. |

---

### 🔍 Analisis Metrik

- **Hit Rate** mengukur apakah ada minimal satu rekomendasi yang relevan. Sistem ini konsisten memberikan hit pada user dengan data valid.
- **Precision@K** mengukur proporsi rekomendasi yang relevan. Nilai tinggi menunjukkan sistem tidak menghasilkan banyak noise.
- **F1-Score** menyeimbangkan precision dan recall. Nilai rendah pada beberapa user lebih disebabkan oleh ketidakseimbangan antara kosakata user vs. item, bukan kualitas rekomendasi yang buruk.
- **DCG & NDCG** mengukur kualitas urutan rekomendasi. NDCG mendekati 1.0 pada mayoritas user menandakan sistem berhasil menempatkan konten paling relevan di posisi teratas.

---

### 💡 Rekomendasi Pengembangan

1. **Perluas dataset** dengan judul-judul film lokal Indonesia agar history user Indonesia dapat diproses secara optimal.
2. **Tambahkan validasi genre** dengan fuzzy matching untuk menangkap preferensi genre yang salah ejaan atau tidak persis sama.
3. **Timbang ulang F1-Score** dengan normalisasi intersection yang lebih adil agar tidak terlalu dipengaruhi ukuran profil user.
4. **Gabungkan dengan Collaborative Filtering** (Hybrid Approach) untuk meningkatkan recall pada kasus cold-start dan genre absurd.
